# Notebook Metadata Bootstrap Example

This notebook demonstrates how to use the `dd_cleaner.notebook_utils` APIs to initialize a notebook session, discover available artifacts, and expose dataset bootstrap metadata through the metadata authority table.


In [7]:
import sys
from pathlib import Path

config_file_name = 'itsm_config.yaml'

# Ensure we import from the local repository source tree, not an installed package.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
repo_root = next((p for p in candidates if (p / config_file_name).exists()), None)
if repo_root is None:
    raise FileNotFoundError(f'Could not locate {config_file_name} in the expected repository paths.')

sys.path.insert(0, str(repo_root / 'src'))
from dd_cleaner.notebook_utils import init_notebook_session, get_metadata_table, get_dataset_metadata

working_dir = repo_root
config_path = repo_root / config_file_name

print('Using workspace:', working_dir)
print('Using config:', config_path)


Using workspace: /home/rajiv/programming/dd-parser-cleaner/tests
Using config: /home/rajiv/programming/dd-parser-cleaner/tests/itsm_config.yaml


In [8]:
coord, artifacts = init_notebook_session(str(working_dir), config_path=str(config_path))

print('Available artifacts:')
display(artifacts)


✅ Notebook session initialized for workspace: /home/rajiv/programming/dd-parser-cleaner/tests

Available Artifacts:

Artifact Name  \
0                         Raw Data   
1                     Cleaned Data   
2                User Cleaned Data   
3             Tagged Entities (DD)   
4  Cleaning Recommendations Report   
5                 Profiling Report   
6                   Handshake File   
7                  Quarantine File   
8               Metadata Authority   

                                         File Name  \
0                          itsm_group_analysis.csv   
1                    itsm_group_analysis_clean.csv   
2             itsm_group_analysis_user_cleaned.csv   
3         itsm_group_analysis_analysis_results.csv   
4                      cleaning_recommendations.md   
5          itsm_group_analysis_profiling_report.md   
6  itsm_group_analysis_parser_cleaner_handshake.md   
7               itsm_group_analysis_quarantine.csv   
8           itsm_group_analysis_metadata_table.csv   

                                            Location  Exists  
0                       data/itsm_group_analysis.csv    True  
1      data/dd_cleaner/itsm_group_analysis_clean.csv    True  
2  data/dd_cleaner/itsm_group_analysis_user_clean...   False  
3  documents/dd_analysis_results/itsm_group_analy...    True  
4   documents/dd_cleaner/cleaning_recommendations.md    True  
5  documents/dd_cleaner/itsm_group_analysis_profi...    True  
6  documents/dd_cleaner/itsm_group_analysis_parse...    True  
7  data/quarantine/itsm_group_analysis_quarantine...   False  
8  data/dd_cleaner/itsm_group_analysis_metadata_t...   False

Available artifacts:


,Artifact Name,File Name,Location,Exists
0,Raw Data,itsm_group_analysis.csv,data/itsm_group_analysis.csv,True
1,Cleaned Data,itsm_group_analysis_clean.csv,data/dd_cleaner/itsm_group_analysis_clean.csv,True
2,User Cleaned Data,itsm_group_analysis_user_cleaned.csv,data/dd_cleaner/itsm_group_analysis_user_clean...,False
3,Tagged Entities (DD),itsm_group_analysis_analysis_results.csv,documents/dd_analysis_results/itsm_group_analy...,True
4,Cleaning Recommendations Report,cleaning_recommendations.md,documents/dd_cleaner/cleaning_recommendations.md,True
5,Profiling Report,itsm_group_analysis_profiling_report.md,documents/dd_cleaner/itsm_group_analysis_profi...,True
6,Handshake File,itsm_group_analysis_parser_cleaner_handshake.md,documents/dd_cleaner/itsm_group_analysis_parse...,True
7,Quarantine File,itsm_group_analysis_quarantine.csv,data/quarantine/itsm_group_analysis_quarantine...,False
8,Metadata Authority,itsm_group_analysis_metadata_table.csv,data/dd_cleaner/itsm_group_analysis_metadata_t...,False


In [9]:
if coord.synchronized_dictionary_path.exists():
    df_metadata = get_metadata_table(coord)
    dataset_metadata = get_dataset_metadata(coord)

    print('Dataset-level bootstrap metadata (separate artifact):')
    display(dataset_metadata)

    print('\nPer-attribute metadata authority table (row-level):')
    print(list(df_metadata.columns))
    display(df_metadata.head())
else:
    print('The cleaner baseline has not been established yet.')
    print('Please run the cleaner pipeline before calling get_metadata_table().')
    print('Example command:')
    print(f'  uv run clean-dataset --config {config_path} --action full')
    print('Then rerun this cell.')


Dataset-level bootstrap metadata (separate artifact):


{'dataset_type': 'event_log',
 'subject': 'support-group',
 'subject_id_attribute': 'number',
 'wide_short_homogeneous': False,
 'wide_short_representative_column': None,
 'graph_type': None,
 'notes': 'Generated by dataset bootstrapping.',
 'use_case_answers': {'use_case': 'To understand time to ticket resolution for different support groups',
  'analysis_objective': "To set clear well-informed ticket resolution time SLA's"}}


Per-attribute metadata authority table (row-level):
['Field Name', 'Data Type', 'Description', 'physical_type', 'logical_type', 'attribute_name', 'provisional_entity_assignment', 'static_dynamic', 'dataset_type', 'subject', 'subject_id_attribute', 'wide_short_homogeneous', 'wide_short_representative_column', 'graph_type', 'notes']


,Field Name,Data Type,Description,physical_type,logical_type,attribute_name,provisional_entity_assignment,static_dynamic,dataset_type,subject,subject_id_attribute,wide_short_homogeneous,wide_short_representative_column,graph_type,notes
0,number,Numeric/String,Incident identifier (24918 different values),int,numeric,number,Incident Management,dynamic,event_log,support-group,number,False,None,None,Generated by dataset bootstrapping.
1,incident_state,Categorical,Eight levels controlling the incident manageme...,int,numeric,incident_state,Incident Management,dynamic,event_log,support-group,number,False,None,None,Generated by dataset bootstrapping.
2,active,Boolean,Attribute that shows whether the record is act...,int,numeric,active,Incident Management,dynamic,event_log,support-group,number,False,None,None,Generated by dataset bootstrapping.
3,reassignment_count,Numeric,Number of times the incident has the group or ...,int,numeric,reassignment_count,Incident Management,dynamic,event_log,support-group,number,False,None,None,Generated by dataset bootstrapping.
4,reopen_count,Numeric,Number of times the incident resolution was re...,int,numeric,reopen_count,Incident Management,dynamic,event_log,support-group,number,False,None,None,Generated by dataset bootstrapping.


## Bootstrap metadata fields exposed by the notebook API

The `get_metadata_table()` function bootstraps the authoritative metadata table from the cleaner's synchronized dictionary and enriches it with dataset bootstrap metadata from `config.yaml`, including fields such as: `dataset_type`, `subject`, `wide_short_homogeneous`, `wide_short_representative_column`, and use-case answers.
